In [1]:
import faiss

In [2]:
import torch

In [3]:
import numpy as np

In [3]:
# pip install pymupdf
# pip install superkmeans
# pip install ollama

In [5]:
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
Built with CUDA: 12.8
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce GTX 1650


In [4]:
from sentence_transformers import SentenceTransformer

D:\Anaconda3\envs\financial_analysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Reading document by each page

- Ensure that we are getting the necessary metadata such as the file name, title, author, pages, page no. 
- This can help in searching based on metadata
- Can also aid in expanding the context of the retrieved chunks before passing to the LLM

In [9]:
# from pathlib import Path
# for pdf_path in Path(r"D:\RAG\pdf files").glob("*.pdf"):
#     print(pdf_path)

In [42]:
import fitz
from langchain_core.documents import Document
from pathlib import Path

docs = []

for pdf_path in Path(r"D:\RAG\pdf files").glob("*.pdf"):
    print(pdf_path)
    pdf = fitz.open(pdf_path)
    pdf_metadata = pdf.metadata
    print(pdf_metadata)
#     toc = pdf.get_toc()
#     print(toc)

    for page_num, page in enumerate(pdf):
        docs.append(
            Document(
                page_content=page.get_text(),
                metadata={
                    "source": str(pdf_path),
                    "filename": pdf_path.name,
                    "page": page_num + 1,
                    "page_count": len(pdf),
                    "title": pdf_metadata.get("title"),
                    "author": pdf_metadata.get("author"),
                },
            )
        )

D:\RAG\pdf files\Adi Polak - Scaling Machine Learning with Spark_ Distributed ML with MLlib, TensorFlow, and PyTorch (2023, O'Reilly Media) - libgen.li.pdf
{'format': 'PDF 1.6', 'title': 'Scaling Machine Learning with Spark', 'author': 'Adi Polak;', 'subject': '', 'keywords': '', 'creator': 'AH CSS Formatter V6.2 MR7 for Linux64 : 6.2.9.19987 (2015/02/24 12:30JST)', 'producer': 'Antenna House PDF Output Library 6.2.658 (Linux64)', 'creationDate': 'D:20230306225228Z', 'modDate': "D:20230308041227-05'00'", 'trapped': '', 'encryption': None}
D:\RAG\pdf files\Ben G Weber - Data Science in Production_ Building Scalable Model Pipelines with Python-Independently published (2020).pdf
{'format': 'PDF 1.4', 'title': 'Data Science in Production: Building Scalable Model Pipelines with Python', 'author': 'Ben G. Weber', 'subject': '', 'keywords': '', 'creator': 'LaTeX via pandoc', 'producer': 'MiKTeX-xdvipdfmx (20190522)', 'creationDate': "D:20191231160314-08'00'", 'modDate': 'D:20200104195850Z', '

In [9]:
import gc
gc.collect()

20

# Chunking 

- Add chunk metadata such as the source, page no, chunk index
- This allows easy search of all the chunks that are a part of a specific page of a PDF

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [11]:
len(chunks)

16261

In [52]:
# Assign chunk indices within each document
doc_chunk_counter = {}

for chunk in chunks:

    doc = chunk.metadata["source"]

    if doc not in doc_chunk_counter:
        doc_chunk_counter[doc] = 0

    chunk.metadata["chunk_index"] = doc_chunk_counter[doc]
    doc_chunk_counter[doc] += 1

In [54]:
chunks[3560]

Document(metadata={'source': 'D:\\RAG\\pdf files\\Christian Hill - Learning Scientific Programming with Python-Cambridge University Press (2020).pdf', 'filename': 'Christian Hill - Learning Scientific Programming with Python-Cambridge University Press (2020).pdf', 'page': 200, 'page_count': 571, 'title': '', 'author': '', 'chunk_index': 969}, page_content='browser.\nThe new notebook document (Figure 5.2) consists of a title bar, a menu bar and a\ntool bar, under which is an IPython prompt where you will type the code and markup\n(e.g. explanatory text and documentation) as a series of cells.\nIn the title bar the name of the ﬁrst notebook you open will probably be “Untitled”;\nclick on it to rename it to something more informative. The menu bar contains options\nfor saving, copying, printing, rearranging and otherwise manipulating the Jupyter Note-')

# Creating embeddings

In [16]:
model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3430.45it/s]


In [65]:
%%time

texts = [c.page_content for c in chunks]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)


Batches: 100%|██████████| 497/497 [06:12<00:00,  1.33it/s]


CPU times: total: 6min 9s
Wall time: 6min 13s


# Saving embeddings in FAISS vector store

In [50]:
# import faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [55]:
metadata = []

for chunk in chunks:
    metadata.append({
    "text": chunk.page_content,
    "source": chunk.metadata["source"],
    "page": chunk.metadata["page"],
    "chunk_index": chunk.metadata["chunk_index"]
})

In [57]:
chunk_lookup = {}

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["chunk_index"]
    )
    chunk_lookup[key] = chunk
    
from collections import defaultdict
page_lookup = defaultdict(list)

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["page"]
    )
    page_lookup[key].append(chunk)

In [61]:
page_lookup[("D:\\RAG\\pdf files\\Adi Polak - Scaling Machine Learning with Spark_ Distributed ML with MLlib, TensorFlow, and PyTorch (2023, O'Reilly Media) - libgen.li.pdf", 3)]

[{'text': 'Praise for Scaling Machine Learning with Spark\nIf there is one book the Spark community has been craving for the last decade, it’s this.\nWriting about the combination of Spark and AI requires broad knowledge, a deep\ntechnical skillset, and the ability to break down complex concepts so they’re easy to\nunderstand. Adi delivers all of this and more while covering big data, AI, and\neverything in between.\n—Andy Petrella, founder at Kensu and author of\nFundamentals of Data Observability (O’Reilly)',
  'source': "D:\\RAG\\pdf files\\Adi Polak - Scaling Machine Learning with Spark_ Distributed ML with MLlib, TensorFlow, and PyTorch (2023, O'Reilly Media) - libgen.li.pdf",
  'page': 3,
  'chunk_index': 7},
 {'text': '—Andy Petrella, founder at Kensu and author of\nFundamentals of Data Observability (O’Reilly)\nScaling Machine Learning with Spark is a wealth of knowledge for data and ML\npractitioners, providing a holistic and creative approach to building end-to-end scalable\n

In [67]:
import pickle

with open("metadata.pkl","wb") as f:
    pickle.dump(metadata,f)

In [68]:
faiss.write_index(index, "docs.index")

In [14]:
index = faiss.read_index("docs.index")

# Retrieval Block

In [45]:
%%time
# query = "What are some good to have metric properties?"
query = "what are some key exercises to strengthen back muscles?"
# query = "who is the CEO of home depot?"

# Questions that fail to retrieve good results
# query = "what is the quarterly performance of walmart?"

query_vector = model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)
# query_vector
D, I = index.search(query_vector, k=50)


CPU times: total: 2.53 s
Wall time: 5.49 s


# Reranker 

In [19]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1656.27it/s]


In [62]:
# %%time
# candidate_chunks = [
#     metadata[i]
#     for i in I[0]
# ]

# pairs = [
#     (query, c["text"])
#     for c in candidate_chunks
# ]

# scores = reranker.predict(pairs)

# sorted_indices = np.argsort(scores)[::-1]
# sorted_indices

In [63]:
# # I_updated = np.array(I[0])[sorted_indices][:5]
# # results = [metadata[i] for i in I_updated]
# # results

# candidate_chunks_dict = {idx: chunk for idx, chunk in enumerate(candidate_chunks)}
# results = [candidate_chunks_dict[idx] for idx in sorted_indices[:20]]
# results

In [64]:
# context = ""

# for r in results:
#     context += f"""
# Source: {r['source']}
# Page: {r['page']}

# {r['text']}

# ----------------
# """
# context

# Weighted ranking 
1. Similarity search - cosine distances
2. Reranking using cross encoder
3. Metadata boost 

In [46]:
%%time
# Retrieve
D, I = index.search(query_vector, k=50)

candidate_chunks = [metadata[i] for i in I[0]]

pairs = [(query, c["text"]) for c in candidate_chunks]

# Reranker scores
rerank_scores = reranker.predict(pairs)

# Normalize FAISS scores
faiss_scores = D[0]
faiss_scores = (faiss_scores - faiss_scores.min()) / (
    faiss_scores.max() - faiss_scores.min() + 1e-8
)

# Normalize reranker scores
rerank_scores = (rerank_scores - rerank_scores.min()) / (
    rerank_scores.max() - rerank_scores.min() + 1e-8
)



CPU times: total: 2.59 s
Wall time: 2.17 s


In [47]:
from collections import Counter
import numpy as np

# Count document occurrences
doc_counts = Counter(
    chunk["source"]
    for chunk in candidate_chunks
)

max_count = max(doc_counts.values())

metadata_scores = []

query_words = set(query.lower().split())

for chunk in candidate_chunks:

    score = 0.0

    # Same-document boost
    score += (
        doc_counts[chunk["source"]]
        / max_count
    )

    # Heading boost (if available)
    if "heading" in chunk:

        overlap = len(
            query_words &
            set(chunk["heading"].lower().split())
        )

        score += overlap / max(1, len(query_words))

    metadata_scores.append(score)

metadata_scores = np.array(metadata_scores)

metadata_scores = (
    metadata_scores
    - metadata_scores.min()
) / (
    metadata_scores.max()
    - metadata_scores.min()
    + 1e-8
)

In [48]:
# Weighted fusion
# alpha = 0.7
# final_scores = alpha * faiss_scores + (1 - alpha) * rerank_scores

final = (
      0.6 * faiss_scores
    + 0.25 * rerank_scores
    + 0.15 * metadata_scores
)

# Sort
ranked = sorted(
    zip(final_scores, candidate_chunks),
    key=lambda x: x[0],
    reverse=True,
)

results = [chunk for _, chunk in ranked[:10]]

# Expanding retrieved chunks

Instead of using only a single chunk, we get all the chunks from that page. This avoids missing any relevant context that may not be present in a chunk

In [68]:
expanded_results = []

for chunk in results:

    key = (
        chunk["source"],
        chunk["page"]
    )

    expanded_results.extend(page_lookup[key])

In [69]:
len(expanded_results)

32

In [70]:
context = ""

for r in expanded_results:
    context += f"""
Source: {r['source']}
Page: {r['page']}

{r['text']}

----------------
"""
context

"\nSource: D:\\RAG\\pdf files\\Matt Furey - Combat Conditioning (1)_text.pdf\nPage: 33\n\nWall Walking \nThis exercise Is another one thal siretches and strengthens all the muscles along the \nspine. E also works the abdorninals as they involuntarily contract when you bend backwards. \nincreased Hexlhilty and strength in the spine goes a long way toward Increasing energy levels \nand Improving overall health, \n1. Stand with your back and heels Hat against the wall, \n2. Take two steps, heel to tos, until you are three feel fram the wall\n\n----------------\n\nSource: D:\\RAG\\pdf files\\Matt Furey - Combat Conditioning (1)_text.pdf\nPage: 33\n\n2. Take two steps, heel to tos, until you are three feel fram the wall \n3. From there, lean backward with your hands stretched above your head, \n4, Slowly move your hands down the wall, Continue walking until the top of your \nhead lightly touches the flow. \n5. Turn to your stomach and stand up again. \n& Do five to ten repetitions. \nz \n2 

# Generation

In [29]:
from ollama import chat

In [71]:
prompt = f"""
You are a helpful assistant answering questions from documents.

Rules:
- Answer only from the provided context.
- If the answer is not present, say "I could not find that information."
- Quote important facts when possible.
- Mention the source document if available.

Context:
{context}

Question:
{query}

Answer:
"""

In [72]:
%%time
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

According to Matt Furey's "Combat Conditioning" text, some key exercises to strengthen back muscles include:

1. Back Bridge: This exercise is considered one of the most effective for strengthening the back muscles, including the abdominals, legs, hips, buttocks, and neck.
2. Kneeling Back Bend: This exercise targets the back and thigh muscles, as well as hip flexors and buttock strength.
3. Reverse Leg Lifts: This exercise develops strength in the abdominals, lower back, and buttocks, while also stretching the lower back.

These exercises are part of Matt Furey's comprehensive program for building strength, flexibility, and overall fitness.
CPU times: total: 78.1 ms
Wall time: 25.2 s
